# 🚀 PFC1: Módulo 2 (HyperNCD) — Predicción sobre `PL7` y `PL13` en Google Colab Pro

Este cuaderno ejecuta el flujo completo que acordamos:
1. **Clonar tu repositorio privado** (`manyorycuevamendoza/PFC1-HyperNCD-Sonata`, rama `colab`).
2. **Celda separada de `git pull`** para refrescar en 1 segundo cualquier cambio que editemos desde Antigravity.
3. **Conectar tu carpeta de Google Drive (`1tvPre2NWeLqvxmac5oleK0RlU2EOAFHw`) y descomprimir (`unzip`)** los pesos entrenados (`.ckpt`) y logs de **HyperNCD (Módulo 2)**.
4. **Instalar el entorno de HyperNCD (`MinkowskiEngine`)** en Colab Pro.
5. **Predecir `PL7` y `PL13` con `predecir_hyperncd.py`**, extrayendo los descriptores semánticos `f_sem` (96 dim) y las clases predichas.

### 🔑 Cómo poner tu Token de GitHub (Solo la primera vez)
En la barra izquierda de Colab, haz clic en la **llave (🔑 Secrets)** → agrega `GITHUB_TOKEN` con tu token `ghp_...` y activa *"Acceso al cuaderno"* (si no lo pones ahí, la Celda 2 te lo pedirá por teclado oculto).

In [ ]:
# @title 1. Verificar GPU asignada en Colab Pro
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# @title 2. Clonar el Repositorio Privado (Rama `colab`)
import os
from getpass import getpass

REPO_OWNER = "manyorycuevamendoza"
REPO_NAME = "PFC1-HyperNCD-Sonata"
BRANCH = "colab"
REPO_DIR = f"/content/{REPO_NAME}"

token = None
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
    if token:
        print("✅ Token leído desde Colab Secrets (GITHUB_TOKEN).")
except Exception:
    pass

if not token:
    token = getpass("Ingresa tu GitHub Personal Access Token (oculto): ").strip()

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} https://{token}@github.com/{REPO_OWNER}/{REPO_NAME}.git {REPO_DIR}
else:
    print(f"El repositorio ya está clonado en {REPO_DIR}.")

%cd {REPO_DIR}
!git status -s
!git log -1 --oneline

---
## 🔄 CELDA SEPARADA: Refrescar cambios desde Git (`git pull`)
Cada vez que editemos código desde **Antigravity** en tu PC, solo dale **Play** a esta celda para actualizar Colab al instante.

In [ ]:
# @title 🔄 Actualizar código desde GitHub (git pull origin colab)
%cd /content/PFC1-HyperNCD-Sonata
!git pull origin colab
!git log -1 --oneline

---
## 📂 3. Conectar Google Drive y Descomprimir Pesos + Logs del Módulo 2 (HyperNCD)
Descarga o monta los archivos de tu carpeta compartida:
`https://drive.google.com/drive/folders/1tvPre2NWeLqvxmac5oleK0RlU2EOAFHw`
y descomprime automáticamente los `.zip` que contengan el `.ckpt`, los logs y los `.npy` de `PL7` y `PL13`.

In [ ]:
# @title 3. Extraer SOLO los pesos (.ckpt) y logs desde hyperncd_export.zip en segundos (sin bajar los 6 GB)
!pip install -q remotezip requests

import os
import glob
import google.auth
from google.auth.transport.requests import Request
from google.colab import drive, auth
from googleapiclient.discovery import build
from remotezip import RemoteZip

drive.mount('/content/drive')
auth.authenticate_user()

FOLDER_ID = "1tvPre2NWeLqvxmac5oleK0RlU2EOAFHw"  # @param {type:"string"}
DRIVE_UNZIP_DIR = "/content/drive/MyDrive/PFC1_HyperNCD_Sonata/datos_m2_descomprimidos"  # @param {type:"string"}
os.makedirs(DRIVE_UNZIP_DIR, exist_ok=True)

# Borrar descarga incompleta de 6 GB si se detuvo antes
zip_parcial = os.path.join(DRIVE_UNZIP_DIR, "hyperncd_export.zip")
if os.path.exists(zip_parcial):
    os.remove(zip_parcial)

ckpts_en_drive = glob.glob(f"{DRIVE_UNZIP_DIR}/**/*.ckpt", recursive=True)

if ckpts_en_drive:
    print(f"⚡ Ya están los pesos (.ckpt) descomprimidos en tu Drive ({DRIVE_UNZIP_DIR}).")
else:
    creds, _ = google.auth.default()
    creds.refresh(Request())
    service = build('drive', 'v3', credentials=creds)

    results = service.files().list(
        q=f"'{FOLDER_ID}' in parents and trashed = false",
        fields="files(id, name, size)",
        supportsAllDrives=True,
        includeItemsFromAllDrives=True,
    ).execute()

    for item in results.get('files', []):
        if item['name'].endswith('.zip'):
            file_id = item['id']
            url = f"https://www.googleapis.com/drive/v3/files/{file_id}?alt=media&supportsAllDrives=true"
            headers = {"Authorization": f"Bearer {creds.token}"}
            print(f"🔍 Inspeccionando {item['name']} remotamente sin descargar los 5.9 GB...")
            with RemoteZip(url, headers=headers) as rz:
                todos = rz.namelist()
                # Filtrar SOLO checkpoints (.ckpt) y logs (evita bajar los 5.5 GB del dataset SemanticPOSS)
                seleccionados = [
                    f for f in todos
                    if not f.endswith('/') and (
                        f.endswith('.ckpt')
                        or 'logs/' in f
                        or f.endswith('.out')
                        or f.endswith('.yaml')
                        or f.endswith('.csv')
                    )
                ]
                print(f"⚡ Extrayendo únicamente {len(seleccionados)} archivos (pesos .ckpt y logs) directo a tu Drive:")
                for member in seleccionados:
                    info = rz.getinfo(member)
                    print(f"  ⬇️ {member} ({info.file_size / (1024*1024):.1f} MB)")
                    rz.extract(member, path=DRIVE_UNZIP_DIR)

print("\n=== Archivos .ckpt (pesos HyperNCD) listos en Drive ===")
ckpts = glob.glob(f"{DRIVE_UNZIP_DIR}/**/*.ckpt", recursive=True)
for c in sorted(set(ckpts)):
    print("  [CKPT]", c)

---
## ⚙️ 4. Instalar Entorno del Módulo 2 (`hyperncd-env` con MinkowskiEngine)
Como `MinkowskiEngine` requiere Python 3.8 + PyTorch 1.10 (`cu113`) + CUDA 11.x (tal como está documentado en `SETUP_KHIPU.md`), esta celda crea el entorno aislado con Miniconda en Colab para que compile sin conflictos con CUDA 12.

In [ ]:
# @title 4. Instalar o Restaurar hyperncd-env + MinkowskiEngine (Guarda caché en Drive como imagen lista)
%%bash
set -e

ENV_CACHE="/content/drive/MyDrive/PFC1_HyperNCD_Sonata/hyperncd_env_cache.tar.gz"

if [ ! -d "/content/miniconda3" ]; then
    echo "[1/4] Instalando Miniconda3 base..."
    wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh
    bash /tmp/miniconda.sh -b -p /content/miniconda3
fi

source /content/miniconda3/etc/profile.d/conda.sh
conda config --set always_yes yes
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main 2>/dev/null || true
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r 2>/dev/null || true

# Si ya existe la imagen del entorno compilada en tu Google Drive, la restauramos en ~20 segundos
if [ -f "$ENV_CACHE" ] && [ ! -d "/content/miniconda3/envs/hyperncd-env" ]; then
    echo "⚡ Encontrado entorno pre-compilado en tu Drive ($ENV_CACHE). Restaurando en ~20 segundos..."
    mkdir -p /content/miniconda3/envs/hyperncd-env
    tar -xzf "$ENV_CACHE" -C /content/miniconda3/envs/hyperncd-env
    /content/miniconda3/envs/hyperncd-env/bin/python -c "import torch, MinkowskiEngine as ME; print('✅ Entorno restaurado desde Drive en segundos! MinkowskiEngine:', ME.__version__)"
    exit 0
fi

if [ ! -d "/content/miniconda3/envs/hyperncd-env" ]; then
    echo "[2/4] Creando entorno hyperncd-env (Python 3.8 + CUDA 11.8 + OpenBLAS)..."
    conda create -y -n hyperncd-env python=3.8 -c conda-forge
    conda install -y -n hyperncd-env -c nvidia/label/cuda-11.8.0 cuda-nvcc cuda-cudart-dev cuda-libraries-dev
    conda install -y -n hyperncd-env -c conda-forge openblas-devel ninja gxx_linux-64=11
fi

conda activate hyperncd-env

echo "[3/4] Instalando PyTorch 1.10.1+cu113 y dependencias de HyperNCD..."
pip install -q torch==1.10.1+cu113 torchvision==0.11.2+cu113 -f https://download.pytorch.org/whl/torch_stable.html
pip install -q pytorch-lightning==1.4.8 torchmetrics==0.7.2 scipy==1.7.3 pyyaml tqdm scikit-learn plotly matplotlib
pip install -q torch-scatter -f https://data.pyg.org/whl/torch-1.10.0+cu113.html

if ! python -c "import MinkowskiEngine" 2>/dev/null; then
    echo "[4/4] Compilando e instalando MinkowskiEngine (solo esta primera vez)..."
    # Asegurar cabecera nvtx3/nvToolsExt.h que pide MinkowskiEngine
    conda install -y -n hyperncd-env -c nvidia/label/cuda-11.8.0 cuda-nvtx-dev 2>/dev/null || true
    if [ ! -f "$CONDA_PREFIX/include/nvtx3/nvToolsExt.h" ]; then
        git clone -q --depth 1 https://github.com/NVIDIA/NVTX.git /tmp/NVTX
        cp -r /tmp/NVTX/c/include/nvtx3 "$CONDA_PREFIX/include/"
        rm -rf /tmp/NVTX
    fi
    if [ ! -d "/content/MinkowskiEngine" ]; then
        git clone -q https://github.com/NVIDIA/MinkowskiEngine.git /content/MinkowskiEngine
    fi
    cd /content/MinkowskiEngine
    export CUDA_HOME="$CONDA_PREFIX"
    export PATH="$CONDA_PREFIX/bin:$PATH"
    export CPLUS_INCLUDE_PATH="$CONDA_PREFIX/include:${CPLUS_INCLUDE_PATH:-}"
    export TORCH_CUDA_ARCH_LIST="7.5;8.0;8.6"
    export MAX_JOBS=4
    export CC="$CONDA_PREFIX/bin/x86_64-conda-linux-gnu-gcc"
    export CXX="$CONDA_PREFIX/bin/x86_64-conda-linux-gnu-g++"
    python setup.py install --blas_include_dirs=${CONDA_PREFIX}/include --blas=openblas
fi

python -c "import torch, MinkowskiEngine as ME; print('✅ MinkowskiEngine listo, versión:', ME.__version__, '| GPU:', torch.cuda.get_device_name(0))"

# Guardar imagen del entorno ya compilado en tu Google Drive para no volver a compilar nunca más
if [ -d "/content/drive/MyDrive" ] && [ ! -f "$ENV_CACHE" ]; then
    echo "💾 Guardando imagen del entorno compilado en tu Drive ($ENV_CACHE) para que la próxima vez cargue en 20 segundos..."
    mkdir -p "$(dirname "$ENV_CACHE")"
    tar -czf "$ENV_CACHE" -C /content/miniconda3/envs/hyperncd-env .
    echo "✅ Imagen del entorno guardada en Drive."
fi

---
## 🧠 5. Predecir `PL7` y `PL13` con el Modelo Entrenado de HyperNCD (`predecir_hyperncd.py`)
Verifica abajo las rutas que imprimió la **Celda 3** para tu `.ckpt` y tus archivos `PL7` y `PL13` (si están en la carpeta descomprimida, el script los detecta automáticamente o puedes pegar la ruta exacta).

In [ ]:
# @title 5. Ejecutar predicción de HyperNCD (Módulo 2) sobre PL7 y PL13
import glob
import os
import zipfile
import tarfile

# Rutas de PL7 y PL13 dentro del repositorio (rama colab)
CKPT_PATH = "AUTO"  # @param {type:"string"}
PL7_NPY_PATH = "/content/PFC1-HyperNCD-Sonata/experimentacion/datos/modulo1_PL7/PL7_SCN0001_preprocesado.npy"  # @param {type:"string"}
PL13_NPY_PATH = "/content/PFC1-HyperNCD-Sonata/experimentacion/datos/modulo3_PL13/PL13_SCN0001.npy"  # @param {type:"string"}
OUTPUT_DIR = "/content/drive/MyDrive/PFC1_HyperNCD_Sonata/resultados_m2"  # @param {type:"string"}

os.makedirs(OUTPUT_DIR, exist_ok=True)

todos_ckpts = (
    glob.glob("/content/drive/**/*.ckpt", recursive=True)
    + glob.glob("/content/**/*.ckpt", recursive=True)
)

if CKPT_PATH == "AUTO":
    best_ckpts = [c for c in todos_ckpts if "best" in os.path.basename(c).lower()]
    CKPT_PATH = best_ckpts[0] if best_ckpts else (todos_ckpts[0] if todos_ckpts else "")

if not os.path.exists(PL7_NPY_PATH):
    todos_npys = glob.glob("/content/**/*.npy", recursive=True) + glob.glob("/content/drive/**/*.npy", recursive=True)
    pl7_matches = [n for n in todos_npys if "pl7" in os.path.basename(n).lower()]
    if pl7_matches:
        PL7_NPY_PATH = pl7_matches[0]

if not os.path.exists(PL13_NPY_PATH):
    todos_npys = glob.glob("/content/**/*.npy", recursive=True) + glob.glob("/content/drive/**/*.npy", recursive=True)
    pl13_matches = [n for n in todos_npys if "pl13" in os.path.basename(n).lower()]
    if pl13_matches:
        PL13_NPY_PATH = pl13_matches[0]

print(f"Usando CKPT : {CKPT_PATH}")
print(f"Usando PL7  : {PL7_NPY_PATH} (existe={os.path.exists(PL7_NPY_PATH)})")
print(f"Usando PL13 : {PL13_NPY_PATH} (existe={os.path.exists(PL13_NPY_PATH)})")

PY_BIN = "/content/miniconda3/envs/hyperncd-env/bin/python"
%cd /content/PFC1-HyperNCD-Sonata

if CKPT_PATH and PL7_NPY_PATH:
    print("\n================ PREDICIENDO PL7 ================")
    !{PY_BIN} predecir_hyperncd.py --input "{PL7_NPY_PATH}" --ckpt "{CKPT_PATH}" --output "{OUTPUT_DIR}/PL7_fsem.npz"
else:
    print("⚠️ No se encontró PL7_NPY_PATH o CKPT_PATH. Verifica la ruta arriba.")

if CKPT_PATH and PL13_NPY_PATH:
    print("\n================ PREDICIENDO PL13 ================")
    !{PY_BIN} predecir_hyperncd.py --input "{PL13_NPY_PATH}" --ckpt "{CKPT_PATH}" --output "{OUTPUT_DIR}/PL13_fsem.npz"
else:
    print("⚠️ No se encontró PL13_NPY_PATH o CKPT_PATH. Verifica la ruta arriba.")

In [ ]:
# @title 6. Generar imágenes, .json, .csv y logs de Módulo 2 (PL7 y PL13) y subirlos a GitHub
from IPython.display import Image, display
import os
import glob
import shutil

PY_BIN = "/content/miniconda3/envs/hyperncd-env/bin/python"
REPO_DIR = "/content/PFC1-HyperNCD-Sonata"
RES_M2_REPO = f"{REPO_DIR}/experimentacion/resultados/modulo2_hyperncd"
os.makedirs(RES_M2_REPO, exist_ok=True)

%cd {REPO_DIR}

for nombre, npy_path in [("PL7", PL7_NPY_PATH), ("PL13", PL13_NPY_PATH)]:
    npz_path = f"{OUTPUT_DIR}/{nombre}_fsem.npz"
    # Copiar el .json y .csv que bota la red (predecir_hyperncd.py) al repo
    for ext in [".json", ".csv", ".log"]:
        src = f"{OUTPUT_DIR}/{nombre}_fsem{ext}"
        if os.path.exists(src):
            shutil.copy2(src, f"{RES_M2_REPO}/{nombre}_prediccion_hyperncd{ext}")

    if npy_path and os.path.exists(npz_path):
        print(f"\n📊 Generando visualización (.png) y métricas (.json) de Módulo 2 para {nombre}...")
        out_base_repo = f"{RES_M2_REPO}/modulo2_{nombre}"
        !{PY_BIN} experimentacion/modulo3_sonata/visualizar_features.py \
            --feats "{npz_path}" --data "{npy_path}" --output "{out_base_repo}" --html

        # Copiar también en experimentacion/resultados/ siguiendo el formato de modulo3_PL13.*
        for ext in [".png", ".json"]:
            if os.path.exists(f"{out_base_repo}{ext}"):
                shutil.copy2(f"{out_base_repo}{ext}", f"{REPO_DIR}/experimentacion/resultados/modulo2_{nombre}{ext}")
                shutil.copy2(f"{out_base_repo}{ext}", f"{OUTPUT_DIR}/modulo2_{nombre}{ext}")

        # Generar visor 3D completo (M1 + M2) en experimentacion/resultados/
        json_m1 = npy_path.replace(".npy", ".json")
        visor_html = f"{REPO_DIR}/experimentacion/resultados/visor_{nombre}_m2.html"
        if os.path.exists(json_m1):
            !{PY_BIN} experimentacion/visor_3d.py --npy "{npy_path}" --json "{json_m1}" --feats "{npz_path}" --output "{visor_html}"
        else:
            !{PY_BIN} experimentacion/visor_3d.py --npy "{npy_path}" --feats "{npz_path}" --output "{visor_html}"

        if os.path.exists(f"{out_base_repo}.png"):
            display(Image(filename=f"{out_base_repo}.png"))

print("\n🚀 Subiendo todos los resultados (.json, .csv, .png) de M2 a GitHub (rama colab)...")
!git config user.name "renzo-felix"
!git config user.email "renzo.felix@utec.edu.pe"
!git add -f experimentacion/resultados/modulo2_hyperncd/*.json experimentacion/resultados/modulo2_hyperncd/*.csv experimentacion/resultados/modulo2_hyperncd/*.png experimentacion/resultados/modulo2_PL7.* experimentacion/resultados/modulo2_PL13.*
!git status -s
!git commit -m "Agregar resultados (.json, .csv e imágenes .png) del Módulo 2 (HyperNCD) para PL7 y PL13" || echo "Sin cambios nuevos para commitear"
!git push origin colab
print("✅ Todo guardado y subido a GitHub en experimentacion/resultados/modulo2_hyperncd/")